In [ ]:
import numpy as np

#node class for decision tree it represents 1 node in the tree
class _Node:
    def __init__(self, *, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature # feature index is for slitting the data
        self.threshold = threshold # value is to use divide the samples into the left and the right child nodes
        self.left = left # References to the Left child node
        self.right = right # References to the Right child node
        self.value = value # The predicted class label if the node is a leaf 

    def is_leaf(self): # if the value is none then it is not a leaf, none is to check if it is a leaf or not
        return self.value is not None


# the training and prediction logic of the decision tree is done here
#those parameters in tandem are used to control the growth of the tree
# prevent overfitting and underfitting 
class ScratchDecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2, max_features=None, seed=42): 
        self.max_depth = max_depth # Maximum depth the tree can reach and grow, None = no limits in the depth
        self.min_samples_split = min_samples_split # Minumum number of samples required to split an internal node, default is 2
        self.max_features = max_features # Number of proportion of features to consider when looking for the best split, None = all features are considered
        self.seed = seed # control the randomnes so the results can be reproduced and easily debugged, current input is 42
        self.root = None # Stores the root node of the trained tree
        self.n_classes = None # Stores of different classes that are available

#Calculates the Gini impurity of a set of labels y, given the number of classes n_classes.
# 
    @staticmethod
    def _gini(y, n_classes):
        m = len(y) #counts the nb of sample in the current node
        if m == 0:
            return 0.0
        counts = np.bincount(y, minlength=n_classes) # count about the nb of samples that are in each class
        probs = counts / m # calculate the probability of each class by dividing the counts by the total number of samples
        return 1.0 - np.sum(probs ** 2) # Gini formula

# Where the best split should happen, it iterates through the features and thresholds
    def _best_split(self, X, y, rng):
        n_samples, n_features = X.shape
        if n_samples < self.min_samples_split:
            return None, None, None, None

        parent_gini = self._gini(y, self.n_classes)
        if parent_gini == 0.0:
            return None, None, None, None

        if self.max_features is None:
            feat_indices = np.arange(n_features)
        elif isinstance(self.max_features, float):
            k = max(1, int(self.max_features * n_features))
            feat_indices = rng.choice(n_features, size=k, replace=False)
        else:
            k = min(n_features, max(1, int(self.max_features)))
            feat_indices = rng.choice(n_features, size=k, replace=False)

        best_gain = 0.0
        best_feat = None
        best_thresh = None
        best_left_mask = None

        for feat in feat_indices:
            col = X[:, feat]
            order = np.argsort(col)
            col_sorted = col[order]
            y_sorted = y[order]

            # Candidate thresholds only where adjacent feature values differ
            diff_idx = np.flatnonzero(col_sorted[1:] != col_sorted[:-1])
            if len(diff_idx) == 0:
                continue

            thresholds = 0.5 * (col_sorted[diff_idx] + col_sorted[diff_idx + 1])

            # Prefix class counts for O(n_samples) split evaluation
            left_counts = np.zeros((n_samples, self.n_classes), dtype=np.int64)
            one_hot = np.eye(self.n_classes, dtype=np.int64)[y_sorted]
            np.cumsum(one_hot, axis=0, out=left_counts)
            total_counts = left_counts[-1]

            for thresh in thresholds:
                # Split index in sorted order
                split_pos = np.searchsorted(col_sorted, thresh, side="right")
                n_left = split_pos
                n_right = n_samples - split_pos
                if n_left == 0 or n_right == 0:
                    continue

                l_cnt = left_counts[split_pos - 1]
                r_cnt = total_counts - l_cnt

                l_gini = 1.0 - np.sum((l_cnt / n_left) ** 2)
                r_gini = 1.0 - np.sum((r_cnt / n_right) ** 2)
                child_gini = (n_left / n_samples) * l_gini + (n_right / n_samples) * r_gini
                gain = parent_gini - child_gini

                if gain > best_gain:
                    best_gain = gain
                    best_feat = int(feat)
                    best_thresh = float(thresh)
                    best_left_mask = X[:, feat] <= thresh

        if best_feat is None or best_gain <= 0.0:
            return None, None, None, None

        return best_feat, best_thresh, best_left_mask, ~best_left_mask

    def _build_tree(self, X, y, depth, rng):
        counts = np.bincount(y, minlength=self.n_classes)
        majority_class = int(np.argmax(counts))

        stop_depth = self.max_depth is not None and depth >= self.max_depth
        stop_samples = len(y) < self.min_samples_split

        if stop_depth or stop_samples:
            return _Node(value=majority_class)

        feat, thresh, left_mask, right_mask = self._best_split(X, y, rng)
        if feat is None:
            return _Node(value=majority_class)

        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1, rng)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1, rng)
        return _Node(feature=feat, threshold=thresh, left=left_child, right=right_child)

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y, dtype=np.int64)
        self.n_classes = int(np.max(y)) + 1
        rng = np.random.default_rng(self.seed)
        self.root = self._build_tree(X, y, depth=0, rng=rng)
        return self

    def _predict_row(self, x, node):
        while not node.is_leaf():
            if x[node.feature] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.value

    def predict(self, X):
        X = np.asarray(X)
        return np.array([self._predict_row(row, self.root) for row in X], dtype=np.int64)